In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('/home/admin/Documents/98_model/src')
sys.path.append('/data01/Documents/98_model/src')

## Import

In [3]:
from project_paths import PROJECT_PATH, DATA_PATH, SRC_PATH
from target_columns.cols_small_y import (
    SMALL_Y_DEFECT_ELECTRODE,
    SMALL_Y_CTQ_ELECTRODE,
    SMALL_Y_DEFECT_WINDING,
    SMALL_Y_CTQ_WINDING,
    SMALL_Y_DEFECT_ASSEMBLY,
    SMALL_Y_CTQ_ASSEMBLY,
    SMALL_Y_DEFECT_WASHING,
    SMALL_Y_CTQ_ACTIVATION
)
import pandas as pd
from data_loader.data_loader import DataLoader
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score, mean_absolute_error
from tqdm import tqdm
import re
import warnings
import copy
from tqdm import trange
import gc
warnings.filterwarnings('ignore')

In [4]:
import polars as pl

## Load data by version

In [5]:
def tmp(x) :
    try : 
        y = x[0]
    except : 
        y = x
    return y

In [6]:
idx_len = 8

In [7]:
data = None

In [8]:
idx = 0
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G322047641,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G322047497,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2V2019386,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI126C1,M2ECOT002,2025-12-18 13:38:30,5AELI126R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322035761,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I152C1,M2ECOT002,2026-02-19 06:29:07,5AF2I152R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G322033237,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELG136C1,M2ECOT001,2025-12-17 05:46:02,5CELG136R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36007,07TCED7LGC0021G322065789,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELH111C1,M2ECOT001,2025-12-17 10:08:44,5CELH111R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36008,07TCED7LGC0021G2W2094917,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI136C1,M2ECOT002,2025-12-18 16:45:37,5AELI136R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G322087686,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I142C1,M2ECOT002,2026-02-19 03:39:50,5AF2I142R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G312001836,59BFC012A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5CELJ195C1,M2ECOT001,2025-12-20 07:32:46,5CELJ195R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
idx = 1
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2W2090235,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI132C1,M2ECOT002,2025-12-18 15:20:26,5AELI132R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G322036001,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2V2044458,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELG187C1,M2ECOT002,2025-12-17 04:54:21,5AELG187R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322043513,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I153C1,M2ECOT002,2026-02-19 07:03:00,5AF2I153R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G322008132,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2J12AC1,M2ECOT002,2026-02-19 14:27:14,5AF2J12AR1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36008,07TCED7LGC0021G322047096,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I143C1,M2ECOT002,2026-02-19 04:13:43,5AF2I143R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G2W2027693,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5CELI1C5C1,M2ECOT001,2025-12-19 08:39:37,5CELI1C5R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G2V2005820,59BFB272A1,5A2EL01002,M2EMIX01602,2025-12-01 13:33:34,5AEL2112C1,M2ECOT002,2025-12-02 10:03:52,5AEL2112R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36011,07TCED7LGC0021G2V2013904,59BFB272A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELI124C1,M2ECOT002,2025-12-18 13:04:31,5AELI124R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
idx = 2
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G322083069,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5CELJ193C1,M2ECOT001,2025-12-20 06:48:21,5CELJ193R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G312002316,59BFC012A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELI134C1,M2ECOT002,2025-12-18 16:09:11,5AELI134R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G322051200,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I141C1,M2ECOT002,2026-02-19 03:39:52,5AF2I141R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G312028989,59BFC012A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I13AC1,M2ECOT002,2026-02-19 03:05:59,5AF2I13AR1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G322011557,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I151C1,M2ECOT002,2026-02-19 06:29:08,5AF2I151R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36008,07TCED7LGC0021G2V2013632,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELH132C1,M2ECOT002,2025-12-17 14:31:04,5AELH132R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G312009649,59BFC012A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELI145C1,M2ECOT002,2025-12-18 19:46:05,5AELI145R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G2V2041311,59BFB272A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELH125C1,M2ECOT001,2025-12-17 15:16:21,5CELH125R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36011,07TCED7LGC0021G2W2045764,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI141C1,M2ECOT002,2025-12-18 18:38:10,5AELI141R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
idx = 3
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2V2030238,59BFB272A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELI153C1,M2ECOT002,2025-12-18 22:05:56,5AELI153R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2W2048357,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELH154C1,M2ECOT002,2025-12-17 17:54:00,5AELH154R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G312026930,59BFC012A1,5C1F223022,M2EMIX01103,2026-02-23 10:55:14,5CF2O145C1,M2ECOT001,2026-02-24 15:21:52,5CF2O145R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322059087,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I142C1,M2ECOT002,2026-02-19 03:39:50,5AF2I142R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G2W2049837,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELJ126C1,M2ECOT002,2025-12-19 13:23:23,5AELJ126R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36008,07TCED7LGC0021G312037987,59BFC012A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I14AC1,M2ECOT002,2026-02-19 05:55:15,5AF2I14AR1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G322045737,59BFC022A1,5C1F223022,M2EMIX01103,2026-02-23 10:55:14,5CF2O177C1,M2ECOT001,2026-02-24 23:39:29,5CF2O177R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G322074500,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5CELJ193C1,M2ECOT001,2025-12-20 06:48:21,5CELJ193R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36011,07TCED7LGC0021G2V2039037,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI158C1,M2ECOT002,2025-12-18 23:30:15,5AELI158R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
idx = 4
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G322045222,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I153C1,M2ECOT002,2026-02-19 07:03:00,5AF2I153R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2W2024515,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELG159C1,M2ECOT002,2025-12-16 20:56:24,5AELG159R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2V2020559,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELG139C1,M2ECOT002,2025-12-16 14:59:12,5AELG139R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G2W2064927,59BFB282A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5CELI156C1,M2ECOT001,2025-12-18 15:17:01,5CELI156R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G322023539,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I147C1,M2ECOT002,2026-02-19 05:21:26,5AF2I147R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36008,07TCED7LGC0021G2W2074856,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELJ175C1,M2ECOT001,2025-12-20 01:16:04,5CELJ175R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G322080796,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELJ121C1,M2ECOT001,2025-12-19 13:31:08,5CELJ121R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G2W2008469,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI124C1,M2ECOT002,2025-12-18 13:04:31,5AELI124R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36011,07TCED7LGC0021G322048404,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I154C1,M2ECOT002,2026-02-19 07:02:59,5AF2I154R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
idx = 5
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G322071635,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2I135C1,M2ECOT002,2026-02-19 01:56:48,5AF2I135R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G322002642,59BFC022A1,5A4F218063,M2EMIX01802,2026-02-18 13:48:48,5CELJ121C1,M2ECOT001,2025-12-19 13:31:08,5CELJ121R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G312002171,59BFC012A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELI136C1,M2ECOT002,2025-12-18 16:45:37,5AELI136R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322091003,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I135C1,M2ECOT002,2026-02-19 01:56:48,5AF2I135R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G2V2015461,59BFB272A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELI137C1,M2ECOT002,2025-12-18 17:21:27,5AELI137R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36008,07TCED7LGC0021G322061457,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ137C1,M2ECOT002,2025-12-19 16:55:31,5AELJ137R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G322075812,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI165C1,M2ECOT002,2025-12-19 01:46:11,5AELI165R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G2V2012945,59BFB272A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELH161C1,M2ECOT001,2025-12-17 22:39:46,5CELH161R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36011,07TCED7LGC0021G312042296,59BFC012A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5CELH111C1,M2ECOT001,2025-12-17 10:08:44,5CELH111R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
idx = 6
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2W2009789,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELG196C1,M2ECOT002,2025-12-17 07:01:53,5AELG196R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G2W2076083,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELG156C1,M2ECOT002,2025-12-16 19:48:29,5AELG156R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2W2000889,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELG196C1,M2ECOT002,2025-12-17 07:01:53,5AELG196R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G2W2074277,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELJ114C1,M2ECOT001,2025-12-19 10:02:04,5CELJ114R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G2V2016272,59BFB272A1,5A1EL01001,M2EMIX01502,2025-12-01 13:02:32,5AEL2111C1,M2ECOT002,2025-12-02 10:03:53,5AEL2111R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36008,07TCED7LGC0021G2V2039921,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELG133C1,M2ECOT002,2025-12-16 13:16:16,5AELG133R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,07TCED7LGC0021G322064674,59BFC022A1,5A4F218063,M2EMIX01802,2026-02-18 13:48:48,5AF2J132C1,M2ECOT002,2026-02-19 14:50:57,5AF2J132R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,07TCED7LGC0021G2W2015144,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5CELJ174C1,M2ECOT001,2025-12-20 00:56:41,5CELJ174R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36011,07TCED7LGC0021G2V2012567,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI118C1,M2ECOT002,2025-12-18 11:18:19,5AELI118R2,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
idx = 7
filename = f'feature_store_v10_n33q_{idx}.parquet'
data_chunk = pd.read_parquet(filename)


for col in [x for x in data_chunk.columns if 'Date' in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))

for col in [x for x in data_chunk.columns if 'ID' in x and 'Degas' not in x] :
    print(data_chunk[col].dtype)
    if 'object' in str(data_chunk[col].dtype) : 
        data_chunk[col] = data_chunk[col].apply(lambda x : tmp(x))
display(data_chunk)

if data is None : 
    data = data_chunk
else :
    data = pd.concat([data, data_chunk], axis=0)
    #data = pd.concat([data, data_chunk], axis=0, ignore_index=True, sort=False)
    del data_chunk
    gc.collect()

datetime64[ns]
object
object
object
object
datetime64[ns]
datetime64[ns]
float64
float64
object
object
object
object
object
object
object
object
object
object
object
object
object
object
object
float64
float64
float64
float64
object
object


,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G2W2031793,59BFB282A1,[],[],NaT,5AF2I128C1,M2ECOT002,2026-02-18 23:37:25,5AF2I128R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G322040380,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5CELJ173C1,M2ECOT001,2025-12-20 00:56:40,5CELJ173R1,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2W2065897,59BFB282A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELI142C1,M2ECOT001,2025-12-18 11:50:57,5CELI142R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322044250,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ147C1,M2ECOT002,2025-12-19 19:39:18,5AELJ147R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G2V2028879,59BFB272A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELG17AC1,M2ECOT002,2025-12-17 02:38:28,5AELG17AR1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36007,None,59BFC012A1,[],[],NaT,5CELG125C1,M2ECOT001,2025-12-17 03:42:21,5CELG125R3,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36008,None,59BFC012A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I146C1,M2ECOT002,2026-02-19 04:47:33,5AF2I146R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,None,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ138C1,M2ECOT002,2025-12-19 16:55:33,5AELJ138R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,None,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2J129C1,M2ECOT002,2026-02-19 14:27:15,5AF2J129R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
data

,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G322047641,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G322047497,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2V2019386,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI126C1,M2ECOT002,2025-12-18 13:38:30,5AELI126R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322035761,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I152C1,M2ECOT002,2026-02-19 06:29:07,5AF2I152R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G322033237,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELG136C1,M2ECOT001,2025-12-17 05:46:02,5CELG136R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36007,None,59BFC012A1,[],[],NaT,5CELG125C1,M2ECOT001,2025-12-17 03:42:21,5CELG125R3,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36008,None,59BFC012A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I146C1,M2ECOT002,2026-02-19 04:47:33,5AF2I146R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,None,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ138C1,M2ECOT002,2025-12-19 16:55:33,5AELJ138R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,None,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2J129C1,M2ECOT002,2026-02-19 14:27:15,5AF2J129R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
data['01_Mixing_Lot ID'] = data['01_Mixing_Lot ID'].astype(str)
data['01_Mixing_Equipment ID'] = data['01_Mixing_Equipment ID'].astype(str)

In [18]:
# data['01_Mixing_Finished Date'] = data['01_Mixing_Finished Date'].astype(str)
# data = data[data['01_Mixing_Finished Date']!="[]"]

In [19]:
def tmp2(x) :
    if x == "[]" :
        y = ""
    else : 
        y = x
    return y

In [20]:
for col_name in [x for x in data.columns if 'Date' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [21]:
for col_name in [x for x in data.columns if 'Lot ID' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [22]:
for col_name in [x for x in data.columns if 'Equipment ID' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [23]:
for col_name in [x for x in data.columns if 'Cell ID' in x] :
    data[col_name] = data[col_name].astype(str)

    data[col_name] = data[col_name].apply(lambda x : tmp2(x))   

In [24]:
data.to_parquet('feature_store_v10_n33q.parquet')

In [25]:
data

,07_Before Degas_Cell ID,BASE_ID,01_Mixing_Lot ID,01_Mixing_Equipment ID,01_Mixing_Finished Date,02_Coating(Back)_Lot ID,02_Coating(Back)_Equipment ID,02_Coating(Back)_Finished Date,03_Roll Pressing_Lot ID,03_Roll Pressing_Equipment ID,...,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 1,DV86__음극 Buffer 수동 장력 * 가속도 WINDING 2,DV86__양극 스풀 장력 * 가속도 WINDING 1,DV86__양극 스풀 장력 * 가속도 WINDING 2,DV88_ABW_용접 시간xVoltage,DV88_CSZ_용접 시간xVoltage,DV88_CRW_용접 시간xVoltage,DV88_CBD_용접 시간xVoltage,DV88_CCR_용접 시간xVoltage,DV88_ELF_용접 시간xVoltage
0,07TCED7LGC0021G322047641,59BFC022A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07TCED7LGC0021G322047497,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5AELJ145C1,M2ECOT002,2025-12-19 19:11:26,5AELJ145R2,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,07TCED7LGC0021G2V2019386,59BFB272A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELI126C1,M2ECOT002,2025-12-18 13:38:30,5AELI126R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,07TCED7LGC0021G322035761,59BFC022A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I152C1,M2ECOT002,2026-02-19 06:29:07,5AF2I152R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,07TCED7LGC0021G322033237,59BFC022A1,5A1EL15012,M2EMIX01502,2025-12-15 16:23:28,5CELG136C1,M2ECOT001,2025-12-17 05:46:02,5CELG136R1,M2EROL012,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36007,None,59BFC012A1,,,NaT,5CELG125C1,M2ECOT001,2025-12-17 03:42:21,5CELG125R3,M2EROL015,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36008,None,59BFC012A1,5A3F218065,M2EMIX01702,2026-02-18 19:33:28,5AF2I146C1,M2ECOT002,2026-02-19 04:47:33,5AF2I146R1,M2EROL011,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36009,None,59BFB282A1,5A2EL15007,M2EMIX01602,2025-12-15 18:32:11,5AELJ138C1,M2ECOT002,2025-12-19 16:55:33,5AELJ138R1,M2EROL010,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36010,None,59BFC022A1,5A1F218055,M2EMIX01502,2026-02-18 16:41:58,5AF2J129C1,M2ECOT002,2026-02-19 14:27:15,5AF2J129R1,M2EROL014,...,250.0,NaN,360.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
